# Classical aspect-based sentiment analysis

## Problem and result at a glance

This project rebuilds part of my MSc Artificial Intelligence coursework at the University of Bath as a reproducible, tested NLP engineering project.

The goal is deliberately practical: turn annotated product reviews into aspect-level sentiment records and product summaries using a classical pipeline. The implementation retains linguistic baselines, a BIO-CRF, a weakly supervised opinion lexicon, and dependency/PMI linking. Component diagnostics remain separate from deployable end-to-end inference.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"

In [2]:
manifest = json.loads((ARTIFACTS / "run_manifest.json").read_text())
corpus = pd.read_csv(ARTIFACTS / "corpus_summary.csv")
baselines = pd.read_csv(ARTIFACTS / "ate_baselines.csv")
crf = pd.read_csv(ARTIFACTS / "crf_metrics.csv")
negation = pd.read_csv(ARTIFACTS / "negation_ablation.csv")
linking = pd.read_csv(ARTIFACTS / "linking_metrics.csv")
end_to_end = pd.read_csv(ARTIFACTS / "end_to_end_metrics.csv")
summaries = pd.read_csv(ARTIFACTS / "product_summaries.csv")

In [3]:
best = baselines.query("match == 'exact'").sort_values("f1").iloc[-1]
crf_dev = crf.query("split == 'dev' and match == 'exact'").iloc[0]
selected = manifest["selected_linker"]
link_dev = linking.query("method == @selected").iloc[0]
e2e = end_to_end.query("match == 'exact'").iloc[0]
display(Markdown(f"**Fresh run.** Best baseline exact F1: **{best.f1:.3f}**; CRF development exact F1: **{crf_dev.f1:.3f}**; selected `{selected}` linker development macro-F1/coverage: **{link_dev.macro_f1:.3f}/{link_dev.coverage:.3f}**; end-to-end test exact F1: **{e2e.f1:.3f}**."))

**Fresh run.** Best baseline exact F1: **0.039**; CRF development exact F1: **0.291**; selected `pmi` linker development macro-F1/coverage: **0.660/0.415**; end-to-end test exact F1: **0.167**.

## System architecture

The diagram keeps the vertical logic of the original GA1 design, but adds the missing model-fitting, development-selection, and held-out inference boundaries. Data frames and dataclasses are represented as ordinary data objects rather than databases.

![Corrected classical ABSA architecture](../assets/pipeline_architecture.svg)

## Data and evaluation boundary

The run uses three Hu–Liu corpora covering 17 products or domains. Raw reviews remain local and are excluded from the repository. Splits are deterministic and grouped by parsed review, so a review cannot cross train, development, and test boundaries.

The inline annotation prefix before `##` is blanked before linguistic processing. Gold aspect phrases are then aligned to actual contiguous mentions in the review sentence. Labels that cannot be aligned remain part of evaluation but cannot supply BIO training spans. This avoids giving the CRF its answer as input.

In [4]:
corpus

,dataset,n_reviews,n_files,n_products,n_labels,positive_labels,negative_labels
0,CustomerReviews-3_domains,2099,3,3,1101,817,284
1,Customer_review_data,313,5,5,2102,1358,744
2,Reviews-9-products,855,9,9,2572,1732,840


In [5]:
alignment = manifest["training_alignment"]
display(Markdown(f"Training BIO alignment: **{alignment['aligned_aspects']:,}/{alignment['total_gold_aspects']:,}** aspects ({alignment['match_rate']:.1%}). Split sizes: `{manifest['split_review_counts']}`."))

Training BIO alignment: **2,547/3,414** aspects (74.6%). Split sizes: `{'dev': 653, 'test': 653, 'train': 1961}`.

## Aspect extraction

Five noun-phrase/frequency pipelines provide interpretable baselines. The supervised model is a linear-chain CRF over BIO tags with lexical, POS, dependency, shape, affix, and neighbouring-token context. Statistical vocabulary and CRF state are fitted on training data only.

Exact matching requires the complete surface span; head matching ignores modifier differences. The gap between the two helps distinguish boundary errors from wholly incorrect aspects.

In [6]:
baseline_exact = baselines.query("match == 'exact'")[
    ["pipeline", "precision", "recall", "f1"]
]
crf_exact = crf.query("split == 'dev' and match == 'exact'").assign(
    pipeline="crf"
)[["pipeline", "precision", "recall", "f1"]]
pd.concat([baseline_exact, crf_exact], ignore_index=True).sort_values("f1", ascending=False)

,pipeline,precision,recall,f1
5,crf,0.477801,0.209066,0.290862
4,statistical_then_linguistic,0.036885,0.041628,0.039113
3,linguistic_then_statistical,0.036349,0.041628,0.038810
1,linguistic,0.022847,0.111933,0.037949
0,raw,0.017100,0.111933,0.029668
2,statistical,0.015935,0.041628,0.023047


![Aspect extraction comparison](../artifacts/figures/ate_comparison.png)

The CRF is more selective than the high-recall candidate baselines, but performance is constrained by the subset of annotations that can be projected onto actual review-text mentions.

## Opinion induction and negation

The opinion lexicon counts adjective and verb lemmas near positive and negative training aspects. A smoothed log-odds sign supplies polarity; low-frequency terms are excluded. At inference time, a left-context negator flips the stored sign. The ablation below changes only that polarity adjustment, not which aspect–opinion links are available.

In [7]:
negation[
    ["method", "negation", "coverage", "accuracy", "macro_f1"]
].sort_values(["method", "negation"])

,method,negation,coverage,accuracy,macro_f1
4,dep,disabled,0.649399,0.720798,0.560448
0,dep,enabled,0.649399,0.740741,0.628386
6,dep->pmi,disabled,0.655874,0.720733,0.559434
2,dep->pmi,enabled,0.655874,0.740480,0.627055
5,pmi,disabled,0.415356,0.779510,0.588283
1,pmi,enabled,0.415356,0.795100,0.660050
7,pmi->dep,disabled,0.655874,0.724965,0.564936
3,pmi->dep,enabled,0.655874,0.755994,0.650132


## Aspect-opinion linking

The corpus has aspect polarity labels but no gold opinion spans or gold aspect–opinion pairs. Linker metrics therefore use **gold aspects** as a component diagnostic: they ask how well polarity linking works when the aspect is already known. They are not end-to-end results.

Four sentence-level strategies are compared: shortest dependency path, PMI, dependency then PMI, and PMI then dependency. Selection uses development macro-F1 first and coverage second.

In [8]:
linking.sort_values(["macro_f1", "coverage"], ascending=False)

,split,method,coverage,accuracy,macro_f1,n_total,n_covered
1,dev,pmi,0.415356,0.795100,0.660050,1081,449
3,dev,pmi->dep,0.655874,0.755994,0.650132,1081,709
0,dev,dep,0.649399,0.740741,0.628386,1081,702
2,dev,dep->pmi,0.655874,0.740480,0.627055,1081,709


![Linker accuracy and coverage](../artifacts/figures/linking_tradeoff.png)

PMI wins the declared development rule by trading lower coverage for more reliable polarity among linked gold aspects.

## End-to-end inference

The deployment path is different from the component diagnostic. The CRF predicts test aspects, the lexicon extracts opinion candidates, and the frozen linker creates aspect–opinion–polarity records. Joint precision, recall, and F1 compare predicted `(aspect span, polarity)` pairs with held-out gold aspect polarity. The coverage column is the proportion of predicted aspects that receive a link.

In [9]:
end_to_end

,split,linker,match,precision,recall,f1,true_positive,false_positive,false_negative,predicted,gold,linked_aspect_coverage
0,test,pmi,exact,0.394737,0.105717,0.166770,135,207,1142,342,1277,0.611993
1,test,pmi,head,0.414706,0.110588,0.174613,141,199,1134,340,1275,0.611993


## Product summaries and error analysis

Product summaries use **predicted aspects**, never gold spans. The full CSV includes occurrence counts and unique-review counts; the latter prevents a repetitive review from dominating a product feature.

![Predicted product summaries](../artifacts/figures/product_summaries.png)

In [10]:
summaries.query("count_mode == 'unique_reviews'").sort_values(
    ["total", "product", "aspect"], ascending=[False, True, True]
).head(15)

,product,aspect,positive,negative,total,count_mode
191,Speaker,speakers,10,1,11,unique_reviews
71,Creative Labs Nomad Jukebox Zen Xtra 40GB,software,7,3,10,unique_reviews
192,Speaker,sound,8,0,8,unique_reviews
72,Creative Labs Nomad Jukebox Zen Xtra 40GB,player,2,4,6,unique_reviews
58,Computer,picture,3,1,4,unique_reviews
73,Creative Labs Nomad Jukebox Zen Xtra 40GB,interface,4,0,4,unique_reviews
74,Creative Labs Nomad Jukebox Zen Xtra 40GB,size,4,0,4,unique_reviews
75,Creative Labs Nomad Jukebox Zen Xtra 40GB,sound,4,0,4,unique_reviews
98,Diaper Champ,diaper champ,3,1,4,unique_reviews
122,Linksys Router,router,3,1,4,unique_reviews


The main failure chain is visible in the aggregates. Unalignable training phrases reduce CRF supervision; missed or fragmented aspects cannot be linked; PMI then leaves many surviving aspects uncovered. The gold-aspect diagnostic isolates the last stage, while the lower joint recall shows what happens when all stages are composed. Surface-form grouping and the lack of annotated opinion spans remain important limits on what can be claimed.

## Conclusions

This case study demonstrates a small but complete NLP system: heterogeneous corpus parsing, leakage-safe fitted state, sequence labelling, weak supervision, relation heuristics, development-only selection, atomic artifact publication, and tested reporting.

The result is not presented as a modern state-of-the-art ABSA model. Its value is the engineering boundary around a transparent classical pipeline, together with an honest account of where annotation alignment, linking coverage, and staged error propagation limit the output.